In [8]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

In [9]:
data = pd.read_csv('walmart_cleaned.csv')

In [10]:
data.head()

,Unnamed: 0,Store,Date,IsHoliday,Dept,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,0,1,2010-02-05,0,1.0,24924.50,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,3,151315
1,1,1,2010-02-05,0,26.0,11737.12,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,3,151315
2,2,1,2010-02-05,0,17.0,13223.76,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,3,151315
3,3,1,2010-02-05,0,45.0,37.44,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,3,151315
4,4,1,2010-02-05,0,28.0,1085.29,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,3,151315


In [11]:
# data = data.drop(['Unnamed: 0','MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5','CPI','Unemployment','Type','Size'
# ,'Temperature',"Fuel_Price",'IsHoliday'],axis=1)


data = data[['Store','Date','Dept','Weekly_Sales','Fuel_Price','IsHoliday','CPI','Unemployment']]

In [12]:
data.head()


,Store,Date,Dept,Weekly_Sales,Fuel_Price,IsHoliday,CPI,Unemployment
0,1,2010-02-05,1.0,24924.50,2.572,0,211.096358,8.106
1,1,2010-02-05,26.0,11737.12,2.572,0,211.096358,8.106
2,1,2010-02-05,17.0,13223.76,2.572,0,211.096358,8.106
3,1,2010-02-05,45.0,37.44,2.572,0,211.096358,8.106
4,1,2010-02-05,28.0,1085.29,2.572,0,211.096358,8.106


In [13]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 417264 entries, 0 to 417263
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         417264 non-null  int64  
 1   Date          417264 non-null  str    
 2   Dept          417264 non-null  float64
 3   Weekly_Sales  417264 non-null  float64
 4   Fuel_Price    417264 non-null  float64
 5   IsHoliday     417264 non-null  int64  
 6   CPI           417264 non-null  float64
 7   Unemployment  417264 non-null  float64
dtypes: float64(5), int64(2), str(1)
memory usage: 25.5 MB


In [14]:
# convert Date type form str to datetime 

data['Date'] = pd.to_datetime(data['Date'])

In [15]:
# to get the create mm-yyyy date column

data['month-year'] = data['Date'].dt.to_period('M').dt.to_timestamp()
data.sort_values('month-year',inplace=True)
data.head()

,Store,Date,Dept,Weekly_Sales,Fuel_Price,IsHoliday,CPI,Unemployment,month-year
0,1,2010-02-05,1.0,24924.50,2.572,0,211.096358,8.106,2010-02-01
29777,4,2010-02-26,67.0,9830.69,2.590,0,126.552286,8.623,2010-02-01
29776,4,2010-02-26,59.0,1505.87,2.590,0,126.552286,8.623,2010-02-01
29775,4,2010-02-26,42.0,9571.95,2.590,0,126.552286,8.623,2010-02-01
29774,4,2010-02-26,83.0,10333.04,2.590,0,126.552286,8.623,2010-02-01


In [16]:
# group sales by date only needed in monthly data

month_sales = data.groupby('month-year')['Weekly_Sales'].sum().reset_index()

month_sales = month_sales.rename(columns={'Weekly_Sales': 'monthly_sales'})

# month_sales.head()

In [17]:
# calculating moving avg for each quarter
    
'''window = 3 -> quarter
            6 -> half year
            12 -> year
            etc...'''
month_sales['moving_avg'] = month_sales['monthly_sales'].rolling(window=3).mean()

In [18]:
# drawing the line of monthly sales and moving average for each quarter

px.line(month_sales, x='month-year',
         y=['monthly_sales','moving_avg'],
         title='Month Sales over time',
         labels={'value' : 'Sales Trend Line'
                 ,'month-year':'Date (Month)'}
                 
        )

In [19]:
# economy  = data.groupby('month-year')[['CPI','Unemployment','Fuel_Price','Weekly_Sales']].mean().reset_index()



agg_rules = {
    'CPI': 'mean',
    'Unemployment': 'mean',
    'Fuel_Price': 'mean',
    'Weekly_Sales': 'mean',
    'IsHoliday': 'max'
}

economy = data.groupby('month-year').agg(agg_rules).reset_index()

In [20]:
# normalizing data if we needed it later

# scaler = StandardScaler()
# economy[['CPI','Unemployment','Fuel_Price','Weekly_Sales']] = scaler.fit_transform(economy[['CPI','Unemployment','Fuel_Price','Weekly_Sales']])

In [21]:
economy.head()

,month-year,CPI,Unemployment,Fuel_Price,Weekly_Sales,IsHoliday
0,2010-02-01,167.452834,8.570455,2.693286,16076.778701,1
1,2010-03-01,167.554940,8.575151,2.786482,15432.626612,0
2,2010-04-01,167.227950,8.446687,2.867585,15745.551340,0
3,2010-05-01,167.291521,8.450495,2.917168,15996.481695,0
4,2010-06-01,167.589941,8.452238,2.788118,16486.250953,0


In [22]:
# highlighting holiday periods on the economic indicators chart

holiday_dates = economy[economy['IsHoliday'] == True]['month-year'].unique()


economy_long = economy.melt(id_vars=['month-year'], 
                            value_vars=['CPI', 'Unemployment', 'Fuel_Price', 'Weekly_Sales'],
                            var_name='Indicator', 
                            value_name='Value')


fig = px.line(economy_long, 
              x='month-year', 
              y='Value', 
              color='Indicator',
              facet_col='Indicator', 
              facet_col_wrap=2,
              title='Economical impact on Sales'
              )

for i in holiday_dates:
    fig.add_vrect(
        x0=i,
        x1=pd.to_datetime(i) + pd.DateOffset(days=20),
        fillcolor="gray",
        opacity=0.3,
        layer="below",
        line_width=0,
    )

fig.update_yaxes(matches=None)

fig.show()


In [23]:
# correlation matrix 

correlation_matrix = economy.corr()


fig = px.imshow(correlation_matrix,
                 labels=dict(color="Correlation"),
                 x=correlation_matrix.columns,
                 y=correlation_matrix.columns,
                 color_continuous_scale="RdBu",
                 zmin=-1, zmax=1,
                 title="Correlation Matrix of Economic Indicators")

fig.show()

In [24]:
stores = data.groupby('Store')['Weekly_Sales'].sum().reset_index()
stores['Store'] = stores['Store'].astype('str') 

In [25]:
stores.head()

,Store,Weekly_Sales
0,1,2.224028e+08
1,2,2.753824e+08
2,3,5.758674e+07
3,4,2.995440e+08
4,5,4.547569e+07


In [26]:
px.bar(
    stores,
    x='Store',
    y='Weekly_Sales',
    labels={'Weekly_Sales':'Total Sales'}
    ,title='Store Sales')

In [27]:
stores_sales = data.groupby(['month-year','Store'])['Weekly_Sales'].sum().reset_index()

In [28]:
px.line(stores_sales,
        x='month-year',
        y='Weekly_Sales',
        color='Store'
        )

In [29]:
px.density_heatmap(stores_sales,
                   x='month-year',
                   y='Store',
                   z='Weekly_Sales',
                   nbinsx=20,
                   nbinsy=45,
                   color_continuous_scale='Viridis',
                   title='Store Sales over Time'
                   )